In [1]:
from datasets import load_dataset

dataset = load_dataset("PatronusAI/financebench")

/home/pranavamurthy-ks/dev/financebench-safety/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dataset

DatasetDict({
    train: Dataset({
        features: ['financebench_id', 'company', 'doc_name', 'question_type', 'question_reasoning', 'domain_question_num', 'question', 'answer', 'justification', 'dataset_subset_label', 'evidence', 'gics_sector', 'doc_type', 'doc_period', 'doc_link'],
        num_rows: 150
    })
})

In [3]:
example = dataset["train"][0]

question = example["question"]
expected_answer = example["answer"]
evidence = example["evidence"][0]["evidence_text"]

In [4]:
example["evidence"][0].keys()

dict_keys(['evidence_text', 'doc_name', 'evidence_page_num', 'evidence_text_full_page'])

In [5]:
print("QUESTION:", question)
print("EXPECTED ANSWER:", expected_answer)
print("EVIDENCE:", evidence[:500])


QUESTION: What is the FY2018 capital expenditure amount (in USD millions) for 3M? Give a response to the question by relying on the details shown in the cash flow statement.
EXPECTED ANSWER: $1577.00
EVIDENCE: Table of Contents 
3M Company and Subsidiaries
Consolidated Statement of Cash Flow s
Years ended December 31
 
(Millions)
 
2018
 
2017
 
2016
 
Cash Flows from Operating Activities
 
 
 
 
 
 
 
Net income including noncontrolling interest
 
$
5,363 
$
4,869 
$
5,058 
Adjustments to reconcile net income including noncontrolling interest to net cash
provided by operating activities
 
 
 
 
 
 
 
Depreciation and amortization
 
 
1,488 
 
1,544 
 
1,474 
Company pension and postretirement contrib


In [6]:
print("Does the evidence contain '1,577'?", "1,577" in evidence)
print("Does the evidence contain '1,828'?", "1,828" in evidence)

Does the evidence contain '1,577'? True
Does the evidence contain '1,828'? True


In [7]:
for term in ["Purchases of property", "Purchases of marketable"]:
    pos = evidence.find(term)
    print(f"\n{evidence[pos:pos + 150]}")


Purchases of property, plant and equipment (PP&E)
 
 
(1,577) 
 
(1,373) 
 
(1,420) 
Proceeds from sale of PP&E and other assets
 
 
262 
 
49 
 
58 


Purchases of marketable securities and investments
 
 
(1,828) 
 
(2,152) 
 
(1,410) 
Proceeds from maturities and sale of marketable securities and i


In [8]:
import requests

response = requests.post(
    "http://localhost:11434/api/generate",
    json={
        "model": "qwen3:4b",
        "prompt": "Explain AI hallucination in 3 simple bullet points.",
        "stream": False
    }
)

result = response.json()
print(result["response"])

Here's a clear 3-bullet explanation of AI hallucination:

*   **AI makes up facts or stories it didn't actually learn** – like inventing false details about historical events, technical specs, or personal experiences.
*   **It's not intentional lying** but a glitch in the model's ability to accurately recall or reason from its training data – the AI "hallucinates" based on patterns it misinterprets.
*   **This breaks trust** because the AI gives answers that seem plausible but are factually incorrect, especially when used for critical tasks like research, medicine, or decisions.


In [9]:
# With Evidence

example = dataset["train"][0]

question = example["question"]
evidence = example["evidence"][0]["evidence_text"]

prompt = f"""
Answer the financial question using ONLY the evidence provided.

Question:
{question}

Evidence:
{evidence}

Instructions:
- Answer using the evidence only.
- If the evidence does not support an answer, say so.
- Do not invent facts.

Answer:
"""

response = requests.post(
    "http://localhost:11434/api/generate",
    json={
        "model": "qwen3:4b",
        "prompt": prompt,
        "stream": False
    }
)

answer = response.json()["response"]
print("QUESTION:", question)
print("\nMODEL ANSWER:", answer)
print("\nEXPECTED ANSWER:", example["answer"])

QUESTION: What is the FY2018 capital expenditure amount (in USD millions) for 3M? Give a response to the question by relying on the details shown in the cash flow statement.

MODEL ANSWER: The capital expenditure amount for FY2018 is reported under "Purchases of property, plant and equipment (PP&E)" in the Cash Flows from Investing Activities section of the cash flow statement. For 2018, this line item shows a cash outflow of $1,577 million. This represents the amount spent on capital expenditures, as it directly corresponds to the purchase of long-term assets.

Thus, the FY2018 capital expenditure amount for 3M is 1577 USD millions.

EXPECTED ANSWER: $1577.00


In [10]:
# Without evidence

prompt = f"""
Answer the following financial question.

Question:
{question}

Answer:
"""

response = requests.post(
    "http://localhost:11434/api/generate",
    json={
        "model": "qwen3:4b",
        "prompt": prompt,
        "stream": False
    }
)

baseline_answer = response.json()["response"]

print("QUESTION:", question)
print("\nBASELINE ANSWER:", baseline_answer)
print("\nEXPECTED ANSWER:", example["answer"])

QUESTION: What is the FY2018 capital expenditure amount (in USD millions) for 3M? Give a response to the question by relying on the details shown in the cash flow statement.

BASELINE ANSWER: Based on the cash flow statement details from **3M Company's FY2018 (fiscal year ending December 31, 2018) annual report** (specifically the 10-K filing for the fiscal year 2018), the capital expenditure (CapEx) amount is:  

**$1,826 million USD**.  

### Key detail from the cash flow statement:
- **Section**: *Cash flows from investing activities*  
- **Line item**: *Capital expenditures*  
- **Amount**: **$1,826 million** (as reported in 3M's FY2018 10-K, Page 50 of the 2018 Annual Report).  

This figure represents the cash outflow for capital expenditures during FY2018, consistent with US GAAP reporting standards. No tax adjustments or conversions were applied, as the value is reported in USD millions as per 3M's disclosures.  

*Note: FY2018 for 3M aligns with the fiscal year ending December

In [11]:
results = [{
    "financebench_id": example["financebench_id"],
    "question": question,
    "expected_answer": example["answer"],
    "baseline_answer": baseline_answer,
    "evidence_answer": answer
}]

import pandas as pd

results_df = pd.DataFrame(results)
results_df

,financebench_id,question,expected_answer,baseline_answer,evidence_answer
0,financebench_id_03029,What is the FY2018 capital expenditure amount ...,$1577.00,Based on the cash flow statement details from ...,The capital expenditure amount for FY2018 is r...


In [12]:
verification_prompt = f"""
You are a financial answer verifier.

Question:
{question}

Evidence:
{evidence}

Proposed answer:
The FY2018 capital expenditure amount was $1,826 million.

Check whether the proposed answer is supported by the evidence.

Return:
VERDICT: SUPPORTED or NOT_SUPPORTED
REASON: Explain briefly using the evidence.
"""

response = requests.post(
    "http://localhost:11434/api/generate",
    json={
        "model": "qwen3:4b",
        "prompt": verification_prompt,
        "stream": False
    }
)

verification_result = response.json()["response"]
print(verification_result)

VERDICT: NOT_SUPPORTED  
REASON: The evidence shows "Purchases of property, plant and equipment (PP&E)" for FY2018 was $1,577 million (a negative value in the cash flow statement indicating outflow). The proposed answer of $1,826 million does not match this line item. The $1,826 figure is close to the "Purchases of marketable securities and investments" ($1,828 million), but this represents debt/investment activity, not capital expenditure (which specifically refers to PP&E purchases). Capital expenditure is explicitly defined by the "Purchases of property, plant and equipment" line in the cash flow statement, not marketable securities.


In [15]:
from collections import Counter

records = dataset["train"]

print("Total questions:", len(records))
print("\nQuestion types:")
print(Counter(row["question_type"] for row in records))

print("\nDocument types:")
print(Counter(row["doc_type"] for row in records))

print("\nQuestions with multiple evidence passages:",
      sum(len(row["evidence"]) > 1 for row in records))

Total questions: 150

Question types:
Counter({'metrics-generated': 50, 'domain-relevant': 50, 'novel-generated': 50})

Document types:
Counter({'10k': 112, '10q': 15, 'Earnings': 14, '8k': 9})

Questions with multiple evidence passages: 35


In [16]:

import requests
import json
import time
from pathlib import Path

records = dataset["train"]
output_file = Path("baseline_results.jsonl")

def ask_qwen(prompt):
    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": "qwen3:4b",
            "prompt": prompt,
            "stream": False,
            "options": {"temperature": 0}
        },
        timeout=300
    )
    response.raise_for_status()
    return response.json()["response"]

# Start fresh for this run
completed = 0

with output_file.open("w", encoding="utf-8") as f:
    for i, example in enumerate(records):
        question = example["question"]

        # Combine every evidence passage for this question
        evidence = "\n\n".join(
            item["evidence_text"]
            for item in example["evidence"]
        )

        # Experiment A: no evidence
        baseline = ask_qwen(
            f"Answer this financial question concisely.\n"
            f"Question: {question}\nAnswer:"
        )

        # Experiment B: evidence provided
        evidence_answer = ask_qwen(
            "Answer the financial question using only the "
            "provided evidence. If the evidence is insufficient, "
            "say so. Do not invent facts.\n\n"
            f"Question: {question}\n\n"
            f"Evidence:\n{evidence}\n\nAnswer:"
        )

        result = {
            "financebench_id": example["financebench_id"],
            "question_type": example["question_type"],
            "question": question,
            "expected_answer": example["answer"],
            "evidence": evidence,
            "baseline_answer": baseline,
            "evidence_answer": evidence_answer
        }

        f.write(json.dumps(result, ensure_ascii=False) + "\n")
        f.flush()

        completed += 1
        print(f"Completed {completed}/{len(records)}")

print(f"\nSaved {completed} questions to {output_file}")


Completed 1/150
Completed 2/150
Completed 3/150
Completed 4/150
Completed 5/150
Completed 6/150
Completed 7/150
Completed 8/150
Completed 9/150
Completed 10/150
Completed 11/150
Completed 12/150
Completed 13/150
Completed 14/150
Completed 15/150
Completed 16/150
Completed 17/150
Completed 18/150
Completed 19/150
Completed 20/150
Completed 21/150
Completed 22/150
Completed 23/150
Completed 24/150
Completed 25/150
Completed 26/150
Completed 27/150
Completed 28/150
Completed 29/150
Completed 30/150
Completed 31/150
Completed 32/150
Completed 33/150
Completed 34/150
Completed 35/150
Completed 36/150
Completed 37/150
Completed 38/150
Completed 39/150
Completed 40/150
Completed 41/150
Completed 42/150
Completed 43/150
Completed 44/150
Completed 45/150
Completed 46/150
Completed 47/150
Completed 48/150
Completed 49/150
Completed 50/150
Completed 51/150
Completed 52/150
Completed 53/150
Completed 54/150
Completed 55/150
Completed 56/150
Completed 57/150
Completed 58/150
Completed 59/150
Comple

In [17]:
import pandas as pd

df = pd.read_json("baseline_results.jsonl", lines=True)

print("Total questions:", len(df))
print("Missing baseline answers:", df["baseline_answer"].isna().sum())
print("Missing evidence answers:", df["evidence_answer"].isna().sum())
print("Duplicate question IDs:", df["financebench_id"].duplicated().sum())

Total questions: 150
Missing baseline answers: 0
Missing evidence answers: 0
Duplicate question IDs: 0


In [18]:
df[[
    "question",
    "expected_answer",
    "baseline_answer",
    "evidence_answer"
]].head(5)

,question,expected_answer,baseline_answer,evidence_answer
0,What is the FY2018 capital expenditure amount ...,$1577.00,"10,500 (USD millions) \n*Source: 3M Company F...",Based solely on the provided cash flow stateme...
1,Assume that you are a public equities analyst....,$8.70,10.0,Based solely on the provided balance sheet evi...
2,Is 3M a capital-intensive business based on FY...,"No, the company is managing its CAPEX and Fixe...",Yes. 3M's FY2022 CapEx/revenue ratio was **~29...,Based solely on the provided evidence for FY20...
3,What drove operating margin change as of FY202...,Operating Margin for 3M in FY2022 has decrease...,For 3M FY2022: \n**Operating margin decreased...,The evidence provided is insufficient to answe...
4,"If we exclude the impact of M&A, which segment...",The consumer segment shrunk by 0.9% organically.,Other segment. (3M reported negative growth of...,"Based solely on the provided evidence, the seg..."


23.10.2026

In [2]:

import pandas as pd
import json
from pathlib import Path

RESULTS_FILE = Path("baseline_results.jsonl")

df = pd.read_json(RESULTS_FILE, lines=True)

print("Number of questions:", len(df))
print("Unique question IDs:", df["financebench_id"].nunique())
print("Columns:", df.columns.tolist())

assert len(df) == 150, "Expected 150 questions"
assert df["financebench_id"].nunique() == 150, "Duplicate question IDs found"

required_columns = [
    "financebench_id",
    "question",
    "expected_answer",
    "evidence",
    "baseline_answer",
    "evidence_answer"
]

assert all(col in df.columns for col in required_columns), \
    "One or more required columns are missing"

print("Dataset validation passed.")


Number of questions: 150
Unique question IDs: 150
Columns: ['financebench_id', 'question_type', 'question', 'expected_answer', 'evidence', 'baseline_answer', 'evidence_answer']
Dataset validation passed.


Evaluation

In [4]:

import pandas as pd

# Load the existing results
df = pd.read_json("baseline_results.jsonl", lines=True)

# Create one row for each baseline response
baseline = df[
    ["financebench_id", "question", "expected_answer",
     "evidence", "baseline_answer"]
].copy()

baseline = baseline.rename(
    columns={"baseline_answer": "response"}
)
baseline["condition"] = "baseline"

# Create one row for each evidence-grounded response
grounded = df[
    ["financebench_id", "question", "expected_answer",
     "evidence", "evidence_answer"]
].copy()

grounded = grounded.rename(
    columns={"evidence_answer": "response"}
)
grounded["condition"] = "evidence_grounded"

# Combine both sets
evaluation = pd.concat(
    [baseline, grounded],
    ignore_index=True
)

print("Total responses:", len(evaluation))
print("\nResponses per condition:")
print(evaluation["condition"].value_counts())

evaluation.head()


Total responses: 300

Responses per condition:
condition
baseline             150
evidence_grounded    150
Name: count, dtype: int64


,financebench_id,question,expected_answer,evidence,response,condition
0,financebench_id_03029,What is the FY2018 capital expenditure amount ...,$1577.00,Table of Contents \n3M Company and Subsidiarie...,"10,500 (USD millions) \n*Source: 3M Company F...",baseline
1,financebench_id_04672,Assume that you are a public equities analyst....,$8.70,Table of Contents \n3M Company and Subsidiarie...,10.0,baseline
2,financebench_id_00499,Is 3M a capital-intensive business based on FY...,"No, the company is managing its CAPEX and Fixe...",3M Company and Subsidiaries\nConsolidated Stat...,Yes. 3M's FY2022 CapEx/revenue ratio was **~29...,baseline
3,financebench_id_01226,What drove operating margin change as of FY202...,Operating Margin for 3M in FY2022 has decrease...,"SG&A, measured as a percent of sales, increase...",For 3M FY2022: \n**Operating margin decreased...,baseline
4,financebench_id_01865,"If we exclude the impact of M&A, which segment...",The consumer segment shrunk by 0.9% organically.,Worldwide Sales Change \nBy Business Segment\n...,Other segment. (3M reported negative growth of...,baseline


In [18]:

def classify_response(numerical_assessment, semantic_assessment, rejected=False):
    """
    numerical_assessment: "Correct", "Incorrect", or "N/A"
    semantic_assessment: "Correct" or "Incorrect"
    rejected: True if the model refused to answer
    """

    # Refusal takes priority over the other categories.
    if rejected:
        return "Rejected"

    numerical = numerical_assessment.strip().lower()
    semantic = semantic_assessment.strip().lower()

    # Validate the inputs so missing labels are not silently classified.
    if numerical not in {"correct", "incorrect", "n/a"}:
        raise ValueError("Numerical assessment must be Correct, Incorrect, or N/A")

    if semantic not in {"correct", "incorrect"}:
        raise ValueError("Semantic assessment must be Correct or Incorrect")

    # If numerical evaluation is not applicable, use semantic correctness.
    if numerical == "n/a":
        return "Correct" if semantic == "correct" else "Incorrect"

    # Both criteria are correct.
    if numerical == "correct" and semantic == "correct":
        return "Correct"

    # Both criteria are wrong.
    if numerical == "incorrect" and semantic == "incorrect":
        return "Incorrect"

    # Exactly one criterion is wrong.
    return "Partially Correct"


test_cases = [
    ("Correct", "Correct", False, "Correct"),
    ("Incorrect", "Correct", False, "Partially Correct"),
    ("Correct", "Incorrect", False, "Partially Correct"),
    ("Incorrect", "Incorrect", False, "Incorrect"),
    ("N/A", "Correct", False, "Correct"),
    ("N/A", "Incorrect", False, "Incorrect"),
    ("N/A", "Incorrect", True, "Rejected"),
]

for numerical, semantic, rejected, expected in test_cases:
    actual = classify_response(numerical, semantic, rejected)
    print(
        f"Numeric={numerical:9} | Semantic={semantic:9} "
        f"| Rejected={rejected!s:5} | Result={actual}"
    )
    assert actual == expected

print("\nAll classification tests passed!")


Numeric=Correct   | Semantic=Correct   | Rejected=False | Result=Correct
Numeric=Incorrect | Semantic=Correct   | Rejected=False | Result=Partially Correct
Numeric=Correct   | Semantic=Incorrect | Rejected=False | Result=Partially Correct
Numeric=Incorrect | Semantic=Incorrect | Rejected=False | Result=Incorrect
Numeric=N/A       | Semantic=Correct   | Rejected=False | Result=Correct
Numeric=N/A       | Semantic=Incorrect | Rejected=False | Result=Incorrect
Numeric=N/A       | Semantic=Incorrect | Rejected=True  | Result=Rejected

All classification tests passed!


In [24]:

def evaluate_one_response(question, expected_answer, evidence, response):

    prompt_text = f"""
    You are evaluating a financial QA response for a research study.

    QUESTION:
    {question}

    REFERENCE ANSWER:
    {expected_answer}

    PROVIDED EVIDENCE:
    {str(evidence)[:10000]}

    MODEL RESPONSE:
    {response}

    Assess these criteria independently.

    1. NUMERICAL ASSESSMENT
    - Correct: all relevant numerical claims are accurate.
    - Incorrect: at least one material numerical claim is inaccurate.
    - N/A: there is no numerical claim to evaluate.
    - Check units, periods, signs, and the financial metric.

    2. SEMANTIC ASSESSMENT
    Judge the response's meaning and qualitative reasoning independently
    of numerical accuracy.

    Mark Correct when the response:
    - addresses the question asked;
    - identifies the correct concept, entity, or financial metric;
    - gives a substantively correct qualitative conclusion and reasoning.

    Mark Incorrect when the response:
    - identifies the wrong segment, metric, or reporting period;
    - gives a materially wrong qualitative conclusion;
    - makes materially false or irrelevant claims.

    IMPORTANT:
    - A wrong number alone does NOT make semantic correctness Incorrect.
    - A response can attempt to answer the question and still be semantically
    incorrect if it identifies the wrong concept or gives a wrong conclusion.
    - Do not infer that a qualitative conclusion is correct merely because
    the response attempts to answer the question.
    - Evaluate each criterion using the reference answer and supplied evidence.
    - If the evidence is insufficient to decide, explain the uncertainty.

    3. REJECTION
    Set rejected=true only when the model refuses or abstains without
    providing a substantive answer.

    Return JSON only:
    {{
    "numerical_assessment": "Correct, Incorrect, or N/A",
    "semantic_assessment": "Correct or Incorrect",
    "rejected": false,
    "reason": "Explain numerical and semantic judgments separately",
    "needs_human_review": true
    }}
    """


    result = requests.post(
        OLLAMA_URL,
        json={
            "model": MODEL_NAME,
            "prompt": prompt_text,
            "stream": False,
            "format": "json",
            "options": {"temperature": 0}
        },
        timeout=180
    )
    result.raise_for_status()
    data = result.json()

    raw_output = data.get("response", "").strip()
    if not raw_output:
        raw_output = data.get("thinking", "").strip()

    start = raw_output.find("{")
    end = raw_output.rfind("}")

    if start == -1 or end == -1:
        raise ValueError(f"No JSON object found: {raw_output[:1000]}")

    assessment = json.loads(raw_output[start:end + 1])

    allowed_numeric = {"correct", "incorrect", "n/a"}
    allowed_semantic = {"correct", "incorrect"}

    if assessment.get("numerical_assessment", "").lower() not in allowed_numeric:
        raise ValueError(f"Invalid numerical label: {assessment}")

    if assessment.get("semantic_assessment", "").lower() not in allowed_semantic:
        raise ValueError(f"Invalid semantic label: {assessment}")

    if not isinstance(assessment.get("rejected"), bool):
        raise ValueError(f"Invalid rejection label: {assessment}")

    return assessment


print("Updated evaluator: numerical and semantic judgments are independent.")


Updated evaluator: numerical and semantic judgments are independent.


In [25]:

final_category = classify_response(
    numerical_assessment=assessment["numerical_assessment"],
    semantic_assessment=assessment["semantic_assessment"],
    rejected=assessment["rejected"]
)

print("Numerical assessment:", assessment["numerical_assessment"])
print("Semantic assessment:", assessment["semantic_assessment"])
print("Rejected:", assessment["rejected"])
print("Final category:", final_category)
print("Reason:", assessment["reason"])


Numerical assessment: Incorrect
Semantic assessment: Incorrect
Rejected: False
Final category: Incorrect
Reason: Numerical assessment: The model claims 3M's quick ratio was 1.8 in Q2 FY2023, but the reference answer states it was 0.96. The quick ratio is calculated as (Cash + Marketable Securities + Accounts Receivable) / Current Liabilities. Using the provided evidence: Cash = $4,258, Marketable Securities = $56, Accounts Receivable = $4,947, Total Current Liabilities = $10,936. Quick ratio = (4,258 + 56 + 4,947) / 10,936 = 9,261 / 10,936 ≈ 0.847, which rounds to 0.85. The model's claim of 1.8 is incorrect.

Semantic assessment: The model states the quick ratio is 'less relevant' for manufacturing companies like 3M, which is qualitatively correct, but the model's conclusion that the quick ratio was 'healthy (>1)' is wrong. The reference answer states the quick ratio was 0.96 (needs improvement to touch 1x), so the model's conclusion that it was 'healthy' is incorrect. The model's reas

In [26]:

# Find the evidence-grounded response for the same question.
sample_grounded = evaluation[
    (evaluation["financebench_id"] == sample["financebench_id"])
    & (evaluation["condition"] == "evidence_grounded")
].iloc[0]

grounded_assessment = evaluate_one_response(
    question=sample_grounded["question"],
    expected_answer=sample_grounded["expected_answer"],
    evidence=sample_grounded["evidence"],
    response=sample_grounded["response"]
)

grounded_category = classify_response(
    numerical_assessment=grounded_assessment["numerical_assessment"],
    semantic_assessment=grounded_assessment["semantic_assessment"],
    rejected=grounded_assessment["rejected"]
)

print("Question ID:", sample_grounded["financebench_id"])
print("Condition:", sample_grounded["condition"])
print("Response:", sample_grounded["response"])
print("\nAssessment:")
print(json.dumps(grounded_assessment, indent=2))
print("\nFinal category:", grounded_category)


Question ID: financebench_id_03029
Condition: evidence_grounded
Response: Based solely on the provided cash flow statement evidence, the FY2018 capital expenditure amount for 3M is **$1,577 million**.

**Reasoning**:  
The cash flow statement explicitly shows under "Cash Flows from Investing Activities" for **2018** (FY2018):  
- **Purchases of property, plant and equipment (PP&E)**: $(1,577) million  

This line item represents the cash outflow for capital expenditures (specifically PP&E purchases) during FY2018. The negative sign indicates a cash outflow, and the amount ($1,577 million) is the direct capital expenditure figure reported in the statement. No other line items in the evidence relate to capital expenditures (e.g., "Acquisitions, net of cash acquired" is $13 million but is not classified as core capital expenditures in this context).  

The evidence is sufficient to answer the question without inference or external knowledge.

Assessment:
{
  "numerical_assessment": "Corre

In [27]:

# Re-evaluate the same five baseline responses using the UPDATED function.
pilot_results_updated = []

for _, row in df.iloc[1:6].iterrows():
    assessment = evaluate_one_response(
        question=row["question"],
        expected_answer=row["expected_answer"],
        evidence=row["evidence"],
        response=row["baseline_answer"]
    )

    category = classify_response(
        numerical_assessment=assessment["numerical_assessment"],
        semantic_assessment=assessment["semantic_assessment"],
        rejected=assessment["rejected"]
    )

    pilot_results_updated.append({
        "financebench_id": row["financebench_id"],
        "numerical_assessment": assessment["numerical_assessment"],
        "semantic_assessment": assessment["semantic_assessment"],
        "rejected": assessment["rejected"],
        "final_category": category,
        "reason": assessment["reason"]
    })

pilot_updated_df = pd.DataFrame(pilot_results_updated)

for _, row in pilot_updated_df.iterrows():
    print("=" * 70)
    print("ID:", row["financebench_id"])
    print("Numerical:", row["numerical_assessment"])
    print("Semantic:", row["semantic_assessment"])
    print("Rejected:", row["rejected"])
    print("Final category:", row["final_category"])
    print("Reason:", row["reason"])


ID: financebench_id_04672
Numerical: Incorrect
Semantic: Incorrect
Rejected: False
Final category: Incorrect
Reason: The reference answer is $8.70 billion for FY2018 net PPNE. The model response says 10.0. The balance sheet shows Property, plant and equipment net as $8,738 million for 2018, which is $8.738 billion. However, the question asks for net PPNE (Property, plant and equipment net) which is a specific term. The model response incorrectly states 10.0. The semantic assessment is incorrect because the model did not identify the correct financial metric (net PPNE) and provided a wrong number. The balance sheet shows Property, plant and equipment net as $8,738 million for 2018, which is $8.738 billion, but the reference answer is $8.70, so the model response is incorrect.
ID: financebench_id_00499
Numerical: Incorrect
Semantic: Incorrect
Rejected: False
Final category: Incorrect
Reason: The model response claims a CapEx/revenue ratio of ~29.6% (CapEx: $10.5B; Revenue: $35.5B). Howev

In [45]:
def evaluate_semantics(question, expected_answer, evidence, response):
    prompt = f"""
You are evaluating a financial question-answering system.

Evaluate SEMANTIC CORRECTNESS ONLY.
Numerical correctness is assessed separately.

QUESTION:
{question}

REFERENCE ANSWER:
{expected_answer}

SOURCE EVIDENCE:
{evidence}

MODEL RESPONSE:
{response}

Rules for semantic assessment:

1. Judge whether the response communicates the correct meaning,
   conclusion, entity, metric, period, and relevant explanation.

2. IGNORE numerical accuracy when making the semantic judgment.
   A wrong numerical value alone must NOT make the semantic
   assessment Incorrect if the response otherwise identifies
   and answers the correct question.

3. SPECIAL RULE FOR NUMERICAL QUESTIONS:
   If the question asks for a numerical value and the response
   gives only a number, assess whether that number is presented
   as an answer to the requested metric and period.

   Do NOT compare the number with the reference answer when
   assigning semantic correctness. That comparison belongs
   exclusively to numerical assessment.

   For example:
   Question: What was FY2018 capital expenditure?
   Reference: 1577
   Response: 10500
   Semantic assessment: Correct
   Reason: The response attempts to answer the requested
   capital expenditure question for the correct period.
   The incorrect amount is handled by numerical assessment.

   This rule applies only when the response does not identify
   the wrong metric, entity, or period, or make another
   substantive semantic error.

4. For questions asking which segment, company, cause, or category,
   identifying the wrong one is semantically Incorrect.

5. For questions asking why something changed, an incorrect or
   unsupported primary explanation is semantically Incorrect.

6. For questions asking for a qualitative conclusion, such as
   whether a business is capital-intensive, the opposite conclusion
   is semantically Incorrect.

7. Do not assume that a response is correct merely because it
   mentions the right topic. Its substantive answer must be correct.

8. Do not penalize a response for a numerical error already handled
   by the numerical assessment. However, if the numerical claim
   changes the substantive conclusion, evaluate that conclusion.

9. Use the reference answer and evidence to assess correctness.
   Do not invent facts.

Return ONLY valid JSON with this structure:
{{
  "semantic_assessment": "Correct" or "Incorrect",
  "reason": "Brief explanation of the semantic judgment",
  "needs_human_review": true or false
}}

Set needs_human_review to true if the reference answer and evidence
appear inconsistent, the response is ambiguous, or the judgment
cannot be made confidently.
"""

    data = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": "qwen3:4b",
            "prompt": prompt,
            "stream": False,
            "format": "json"
        },
        timeout=180
    ).json()

    raw = (
        data.get("response", "").strip()
        or data.get("thinking", "").strip()
    )

    start = raw.find("{")
    end = raw.rfind("}")

    if start == -1 or end == -1:
        raise ValueError(f"No JSON found in model output: {raw}")

    result = json.loads(raw[start:end + 1])

    if result.get("semantic_assessment") not in {
        "Correct", "Incorrect"
    }:
        raise ValueError(f"Invalid semantic label: {result}")

    return result


print("Updated semantic evaluator.")


Updated semantic evaluator.


In [ ]:

test_row = df[
    df["financebench_id"] == "financebench_id_04672"
].iloc[0]

semantic_result = evaluate_semantics(
    question=test_row["question"],
    expected_answer=test_row["expected_answer"],
    evidence=test_row["evidence"],
    response=test_row["baseline_answer"]
)

print(json.dumps(semantic_result, indent=2))

In [ ]:
# Numerical result from the earlier assessment of this same response.
numerical_result = "Incorrect"

# Semantic result from your latest separate semantic evaluator.
semantic_result_label = semantic_result["semantic_assessment"]

# Combine them using your tested classification function.
final_result = classify_response(
    numerical_assessment=numerical_result,
    semantic_assessment=semantic_result_label,
    rejected=False
)

print("Numerical:", numerical_result)
print("Semantic:", semantic_result_label)
print("Final category:", final_result)


In [51]:

def evaluate_numerics(question, expected_answer, evidence, response):
    numerical_prompt = f"""
You are evaluating ONLY numerical correctness in a financial QA response.

QUESTION:
{question}

REFERENCE ANSWER:
{expected_answer}

EVIDENCE:
{str(evidence)[:10000]}

MODEL RESPONSE:
{response}

Assess numerical correctness only.

Rules:
1. Identify the numerical value(s) required to answer the question.
2. Check values against the reference answer and evidence.
3. Check units, scale (thousands, millions, billions), reporting period,
   percentage versus percentage points, and positive/negative signs.
4. A value is not correct merely because it looks plausible.
5. Do not call a materially different value a rounding difference.
6. Do not judge semantic correctness or the quality of qualitative reasoning.
7. If the response has no numerical claim that needs evaluation, use "N/A".
8. If the evidence and reference conflict or are insufficient, explain that
   uncertainty rather than inventing a value.
9. Set needs_human_review to true ONLY if:
   - The reference answer conflicts with the evidence.
   - The evidence is insufficient to verify the numerical answer.
   - Units, interpretation, or the required calculation are genuinely ambiguous.

10. Set needs_human_review to false if the evidence supports a clear
    comparison and the response is plainly Correct or Incorrect.

11. Do not flag an answer for review merely because its numerical
    value is incorrect.

Return JSON only:
{{
  "numerical_assessment": "Correct, Incorrect, or N/A",
  "reason": "Explain the numerical judgment",
  "needs_human_review": true or false
}}
"""

    result = requests.post(
        OLLAMA_URL,
        json={
            "model": MODEL_NAME,
            "prompt": numerical_prompt,
            "stream": False,
            "format": "json",
            "options": {"temperature": 0}
        },
        timeout=180
    )

    result.raise_for_status()
    data = result.json()

    raw = data.get("response", "").strip()
    if not raw:
        raw = data.get("thinking", "").strip()

    start = raw.find("{")
    end = raw.rfind("}")

    if start == -1 or end == -1:
        raise ValueError(f"No JSON object found: {raw[:1000]}")

    assessment = json.loads(raw[start:end + 1])

    if assessment.get("numerical_assessment", "").lower() not in {
        "correct", "incorrect", "n/a"
    }:
        raise ValueError(f"Invalid numerical label: {assessment}")

    return assessment


print("Separate numerical evaluator is ready.")

Separate numerical evaluator is ready.


In [34]:

numerical_result_detail = evaluate_numerics(
    question=test_row["question"],
    expected_answer=test_row["expected_answer"],
    evidence=test_row["evidence"],
    response=test_row["baseline_answer"]
)

print(json.dumps(numerical_result_detail, indent=2))


{
  "numerical_assessment": "Incorrect",
  "reason": "The reference answer states $8.70 billion for FY2018 net PPNE. The evidence shows Property, plant and equipment net as $8,738 million (or $8.738 billion) for 2018. The model response says 10.0, which is incorrect. The correct value should be approximately $8.74 billion, which rounds to $8.70 billion as per the reference answer. The model response's 10.0 is not numerically correct.",
  "needs_human_review": true
}


In [35]:

# Evaluate the same baseline response independently.
numeric_check = evaluate_numerics(
    question=test_row["question"],
    expected_answer=test_row["expected_answer"],
    evidence=test_row["evidence"],
    response=test_row["baseline_answer"]
)

semantic_check = evaluate_semantics(
    question=test_row["question"],
    expected_answer=test_row["expected_answer"],
    evidence=test_row["evidence"],
    response=test_row["baseline_answer"]
)

# Determine whether the model refused to answer.
# This is a preliminary rule; human review is still needed.
response_text = str(test_row["baseline_answer"]).strip().lower()

rejection_phrases = [
    "i cannot answer",
    "i can't answer",
    "insufficient information",
    "insufficient evidence",
    "i cannot determine",
    "i'm unable to answer"
]

is_rejected = (
    any(phrase in response_text for phrase in rejection_phrases)
    and len(response_text) < 500
)

# Combine the independent judgments.
category = classify_response(
    numerical_assessment=numeric_check["numerical_assessment"],
    semantic_assessment=semantic_check["semantic_assessment"],
    rejected=is_rejected
)

print("Numerical assessment:", numeric_check["numerical_assessment"])
print("Numerical reason:", numeric_check["reason"])
print()
print("Semantic assessment:", semantic_check["semantic_assessment"])
print("Semantic reason:", semantic_check["reason"])
print()
print("Rejected:", is_rejected)
print("FINAL CATEGORY:", category)


Numerical assessment: Incorrect
Numerical reason: The reference answer states $8.70 billion for FY2018 net PPNE. The evidence shows Property, plant and equipment net as $8,738 million (or $8.738 billion) for 2018. The model response says 10.0, which is incorrect. The correct value should be approximately $8.74 billion, which rounds to $8.70 billion as per the reference answer. The model response's 10.0 is not numerically correct.

Semantic assessment: Correct
Semantic reason: The question asks for the year end FY2018 net PPNE for 3M using the balance sheet. The reference answer is $8.70 billion. The model response provides $10.0, which is a numerical value but the semantic assessment focuses on whether the response addresses the correct question, company, metric, and period. The balance sheet shows 'Property, plant and equipment net' as $8,738 million for 2018, which is $8.738 billion. The model response of $10.0 is a numerical approximation but the semantic correctness is about the co

In [36]:

# Test five baseline responses with both independent evaluators.
pilot_ids = [
    "financebench_id_03029",
    "financebench_id_04672",
    "financebench_id_00499",
    "financebench_id_01226",
    "financebench_id_01865"
]

combined_pilot = []

for question_id in pilot_ids:
    row = df[df["financebench_id"] == question_id].iloc[0]

    numeric_check = evaluate_numerics(
        question=row["question"],
        expected_answer=row["expected_answer"],
        evidence=row["evidence"],
        response=row["baseline_answer"]
    )

    semantic_check = evaluate_semantics(
        question=row["question"],
        expected_answer=row["expected_answer"],
        evidence=row["evidence"],
        response=row["baseline_answer"]
    )

    # Keep refusal detection separate for human review.
    response_text = str(row["baseline_answer"]).strip().lower()

    rejection_phrases = [
        "i cannot answer",
        "i can't answer",
        "i'm unable to answer",
        "insufficient information",
        "insufficient evidence",
        "i cannot determine"
    ]

    is_rejected = (
        any(phrase in response_text for phrase in rejection_phrases)
        and len(response_text) < 500
    )

    category = classify_response(
        numerical_assessment=numeric_check["numerical_assessment"],
        semantic_assessment=semantic_check["semantic_assessment"],
        rejected=is_rejected
    )

    combined_pilot.append({
        "financebench_id": question_id,
        "numerical_assessment": numeric_check["numerical_assessment"],
        "semantic_assessment": semantic_check["semantic_assessment"],
        "rejected": is_rejected,
        "final_category": category,
        "numerical_reason": numeric_check["reason"],
        "semantic_reason": semantic_check["reason"]
    })

combined_pilot_df = pd.DataFrame(combined_pilot)

print(
    combined_pilot_df[
        [
            "financebench_id",
            "numerical_assessment",
            "semantic_assessment",
            "rejected",
            "final_category"
        ]
    ].to_string(index=False)
)


      financebench_id numerical_assessment semantic_assessment  rejected    final_category
financebench_id_03029            Incorrect             Correct     False Partially Correct
financebench_id_04672            Incorrect             Correct     False Partially Correct
financebench_id_00499            Incorrect           Incorrect     False         Incorrect
financebench_id_01226            Incorrect           Incorrect     False         Incorrect
financebench_id_01865            Incorrect             Correct     False Partially Correct


In [39]:

row = df[
    df["financebench_id"] == "financebench_id_01865"
].iloc[0]

semantic_test = evaluate_semantics(
    question=row["question"],
    expected_answer=row["expected_answer"],
    evidence=row["evidence"],
    response=row["baseline_answer"]
)

print(json.dumps(semantic_test, indent=2))


{
  "semantic_assessment": "Incorrect",
  "reason": "The question asks which segment dragged down 3M's overall growth in 2022 when excluding M&A impacts. The reference answer states that the consumer segment shrunk by 0.9% organically. However, the model response identifies 'Other segment' as the one with negative growth of -1.5% excluding M. The evidence table shows that the Consumer segment has a negative organic sales change of -0.9%, while the 'Other' segment is not explicitly listed in the table. The model response incorrectly identifies the segment as 'Other' instead of 'Consumer', which directly contradicts the reference answer and the evidence. The question specifically asks for the segment that dragged down growth, and the reference clearly points to Consumer, not Other. Therefore, the semantic correctness is Incorrect due to the wrong segment identification.",
  "needs_human_review": true
}


In [40]:

combined_pilot = []

pilot_ids = [
    "financebench_id_03029",
    "financebench_id_04672",
    "financebench_id_00499",
    "financebench_id_01226",
    "financebench_id_01865"
]

for question_id in pilot_ids:
    row = df[df["financebench_id"] == question_id].iloc[0]

    numeric_check = evaluate_numerics(
        question=row["question"],
        expected_answer=row["expected_answer"],
        evidence=row["evidence"],
        response=row["baseline_answer"]
    )

    semantic_check = evaluate_semantics(
        question=row["question"],
        expected_answer=row["expected_answer"],
        evidence=row["evidence"],
        response=row["baseline_answer"]
    )

    response_text = str(row["baseline_answer"]).strip().lower()
    rejection_phrases = [
        "i cannot answer",
        "i can't answer",
        "i'm unable to answer",
        "insufficient information",
        "insufficient evidence",
        "i cannot determine"
    ]

    is_rejected = (
        any(phrase in response_text for phrase in rejection_phrases)
        and len(response_text) < 500
    )

    category = classify_response(
        numerical_assessment=numeric_check["numerical_assessment"],
        semantic_assessment=semantic_check["semantic_assessment"],
        rejected=is_rejected
    )

    combined_pilot.append({
        "financebench_id": question_id,
        "numerical_assessment": numeric_check["numerical_assessment"],
        "semantic_assessment": semantic_check["semantic_assessment"],
        "rejected": is_rejected,
        "final_category": category
    })

combined_pilot_df = pd.DataFrame(combined_pilot)

print(combined_pilot_df.to_string(index=False))

      financebench_id numerical_assessment semantic_assessment  rejected final_category
financebench_id_03029            Incorrect           Incorrect     False      Incorrect
financebench_id_04672            Incorrect           Incorrect     False      Incorrect
financebench_id_00499            Incorrect           Incorrect     False      Incorrect
financebench_id_01226            Incorrect           Incorrect     False      Incorrect
financebench_id_01865            Incorrect           Incorrect     False      Incorrect


In [41]:
for question_id in pilot_ids:
    row = df[df["financebench_id"] == question_id].iloc[0]

    print("=" * 100)
    print("ID:", question_id)
    print("\nQUESTION:")
    print(row["question"])

    print("\nEXPECTED ANSWER:")
    print(row["expected_answer"])

    print("\nBASELINE RESPONSE:")
    print(row["baseline_answer"])

    print("\nEVIDENCE:")
    print(str(row["evidence"])[:1200])

ID: financebench_id_03029

QUESTION:
What is the FY2018 capital expenditure amount (in USD millions) for 3M? Give a response to the question by relying on the details shown in the cash flow statement.

EXPECTED ANSWER:
$1577.00

BASELINE RESPONSE:
10,500 (USD millions)  
*Source: 3M Company FY2018 Cash Flow Statement (Investing Activities section)*

EVIDENCE:
Table of Contents 
3M Company and Subsidiaries
Consolidated Statement of Cash Flow s
Years ended December 31
 
(Millions)
 
2018
 
2017
 
2016
 
Cash Flows from Operating Activities
 
 
 
 
 
 
 
Net income including noncontrolling interest
 
$
5,363 
$
4,869 
$
5,058 
Adjustments to reconcile net income including noncontrolling interest to net cash
provided by operating activities
 
 
 
 
 
 
 
Depreciation and amortization
 
 
1,488 
 
1,544 
 
1,474 
Company pension and postretirement contributions
 
 
(370) 
 
(967) 
 
(383) 
Company pension and postretirement expense
 
 
410 
 
334 
 
250 
Stock-based compensation expense
 
 

In [42]:
for question_id in pilot_ids:
    row = df[df["financebench_id"] == question_id].iloc[0]

    numeric_check = evaluate_numerics(
        question=row["question"],
        expected_answer=row["expected_answer"],
        evidence=row["evidence"],
        response=row["baseline_answer"]
    )

    semantic_check = evaluate_semantics(
        question=row["question"],
        expected_answer=row["expected_answer"],
        evidence=row["evidence"],
        response=row["baseline_answer"]
    )

    print("=" * 80)
    print(question_id)
    print("\nNUMERICAL:")
    print(json.dumps(numeric_check, indent=2))
    print("\nSEMANTIC:")
    print(json.dumps(semantic_check, indent=2))

financebench_id_03029

NUMERICAL:
{
  "numerical_assessment": "Incorrect",
  "reason": "The reference answer states $1577.00 USD millions for FY2018 capital expenditure. The model response claims 10,500 USD millions. The evidence shows 'Purchases of property, plant and equipment (PP&E)' for 2018 as -$1,577 million (negative sign indicates cash outflow). The model response incorrectly states 10,500 instead of 1577. The value 10,500 is not present in the evidence and does not match the reference answer.",
  "needs_human_review": true
}

SEMANTIC:
{
  "semantic_assessment": "Incorrect",
  "reason": "The response states '10,500 (USD millions)' but the reference answer is $1577.00. The model response incorrectly identifies the capital expenditure amount as 10,500 USD millions, whereas the correct value from the cash flow statement is $1,577 million. The model response also incorrectly references the 'Investing Activities section' as the source, but the actual capital expenditure amount (Pur

In [46]:
test_ids = [
    "financebench_id_03029",
    "financebench_id_01865"
]

for question_id in test_ids:
    row = df[df["financebench_id"] == question_id].iloc[0]

    result = evaluate_semantics(
        question=row["question"],
        expected_answer=row["expected_answer"],
        evidence=row["evidence"],
        response=row["baseline_answer"]
    )

    print("=" * 70)
    print(question_id)
    print(json.dumps(result, indent=2))

financebench_id_03029
{
  "semantic_assessment": "Correct",
  "reason": "The response correctly identifies the capital expenditure amount for 3M in FY2018 from the Investing Activities section of the cash flow statement, specifying it as $10,500 (USD millions). The response mentions the correct entity (3M Company), metric (capital expenditure), period (FY2018), and source (Investing Activities section).",
  "needs_human_review": false
}
financebench_id_01865
{
  "semantic_assessment": "Incorrect",
  "reason": "The response claims the 'Other segment' dragged down growth, but the reference answer specifies the consumer segment as shrinking by 0.9% organically. The evidence shows the consumer segment had an organic sales change of -0.9% (not 'Other segment'), and the question asks which segment dragged down growth when excluding M&A impacts.",
  "needs_human_review": false
}


In [47]:
combined_pilot = []

for question_id in pilot_ids:
    row = df[df["financebench_id"] == question_id].iloc[0]

    numeric_check = evaluate_numerics(
        question=row["question"],
        expected_answer=row["expected_answer"],
        evidence=row["evidence"],
        response=row["baseline_answer"]
    )

    semantic_check = evaluate_semantics(
        question=row["question"],
        expected_answer=row["expected_answer"],
        evidence=row["evidence"],
        response=row["baseline_answer"]
    )

    category = classify_response(
        numerical_assessment=numeric_check["numerical_assessment"],
        semantic_assessment=semantic_check["semantic_assessment"],
        rejected=False
    )

    combined_pilot.append({
        "financebench_id": question_id,
        "numerical_assessment": numeric_check["numerical_assessment"],
        "semantic_assessment": semantic_check["semantic_assessment"],
        "final_category": category,
        "numerical_reason": numeric_check["reason"],
        "semantic_reason": semantic_check["reason"]
    })

combined_pilot_df = pd.DataFrame(combined_pilot)

print(
    combined_pilot_df[
        [
            "financebench_id",
            "numerical_assessment",
            "semantic_assessment",
            "final_category"
        ]
    ].to_string(index=False)
)

      financebench_id numerical_assessment semantic_assessment    final_category
financebench_id_03029            Incorrect             Correct Partially Correct
financebench_id_04672            Incorrect             Correct Partially Correct
financebench_id_00499            Incorrect           Incorrect         Incorrect
financebench_id_01226            Incorrect           Incorrect         Incorrect
financebench_id_01865            Incorrect           Incorrect         Incorrect


In [48]:
print("Numerical review flags:")
print(
    combined_pilot_df["numerical_reason"]
    .isna()
    .sum()
)

print("\nPilot final-category counts:")
print(
    combined_pilot_df["final_category"]
    .value_counts()
)

Numerical review flags:
0

Pilot final-category counts:
final_category
Incorrect            3
Partially Correct    2
Name: count, dtype: int64


In [53]:
for question_id in pilot_ids:
    row = df[df["financebench_id"] == question_id].iloc[0]

    numeric_check = evaluate_numerics(
        question=row["question"],
        expected_answer=row["expected_answer"],
        evidence=row["evidence"],
        response=row["baseline_answer"]
    )

    semantic_check = evaluate_semantics(
        question=row["question"],
        expected_answer=row["expected_answer"],
        evidence=row["evidence"],
        response=row["baseline_answer"]
    )

    print(
        question_id,
        "| Numerical review:", numeric_check.get("needs_human_review"),
        "| Semantic review:", semantic_check.get("needs_human_review")
    )

financebench_id_03029 | Numerical review: False | Semantic review: False
financebench_id_04672 | Numerical review: False | Semantic review: False
financebench_id_00499 | Numerical review: False | Semantic review: False
financebench_id_01226 | Numerical review: False | Semantic review: False
financebench_id_01865 | Numerical review: False | Semantic review: False


In [52]:
row = df[df["financebench_id"] == "financebench_id_03029"].iloc[0]

numeric_check = evaluate_numerics(
    question=row["question"],
    expected_answer=row["expected_answer"],
    evidence=row["evidence"],
    response=row["baseline_answer"]
)

print(json.dumps(numeric_check, indent=2))

{
  "numerical_assessment": "Incorrect",
  "reason": "The reference answer states $1577.00 USD millions for FY2018 capital expenditure. The evidence shows 'Purchases of property, plant and equipment (PP&E)' as -$1,577 million for 2018. The model response claims 10,500 USD millions, which is incorrect. The correct value is $1577 million (not 10,500).",
  "needs_human_review": false
}


In [54]:
import json
import os
import time
import pandas as pd

OUTPUT_FILE = "evaluation_results.jsonl"

# Each question is evaluated twice: baseline and evidence-grounded.
conditions = [
    ("baseline", "baseline_answer"),
    ("evidence_grounded", "evidence_answer")
]

# Resume support: load previously completed records.
completed_keys = set()

if os.path.exists(OUTPUT_FILE):
    with open(OUTPUT_FILE, "r", encoding="utf-8") as f:
        for line in f:
            try:
                item = json.loads(line)
                if item.get("status") == "completed":
                    completed_keys.add(
                        (item["financebench_id"], item["condition"])
                    )
            except (json.JSONDecodeError, KeyError):
                continue

total = len(df) * len(conditions)
already_done = len(completed_keys)

print(f"Total responses: {total}")
print(f"Already completed: {already_done}")
print(f"Remaining: {total - already_done}")
print("Output file:", OUTPUT_FILE)

# Append results so completed work survives interruptions.
with open(OUTPUT_FILE, "a", encoding="utf-8") as out:
    processed = already_done

    for _, row in df.iterrows():
        question_id = str(row["financebench_id"])

        for condition, answer_column in conditions:
            key = (question_id, condition)

            if key in completed_keys:
                continue

            response = str(row[answer_column])

            record = {
                "financebench_id": question_id,
                "condition": condition,
                "question": row["question"],
                "expected_answer": row["expected_answer"],
                "evidence": row["evidence"],
                "response": response,
                "status": "error"
            }

            try:
                numeric = evaluate_numerics(
                    question=row["question"],
                    expected_answer=row["expected_answer"],
                    evidence=row["evidence"],
                    response=response
                )

                semantic = evaluate_semantics(
                    question=row["question"],
                    expected_answer=row["expected_answer"],
                    evidence=row["evidence"],
                    response=response
                )

                # Basic refusal detection; flagged cases can be reviewed later.
                response_lower = response.lower()
                refusal_phrases = [
                    "i cannot answer",
                    "i can't answer",
                    "i am unable to answer",
                    "i'm unable to answer",
                    "insufficient information",
                    "insufficient evidence",
                    "i cannot determine",
                    "i can't determine"
                ]

                is_rejected = any(
                    phrase in response_lower
                    for phrase in refusal_phrases
                ) and len(response) < 500

                final_category = classify_response(
                    numerical_assessment=numeric[
                        "numerical_assessment"
                    ],
                    semantic_assessment=semantic[
                        "semantic_assessment"
                    ],
                    rejected=is_rejected
                )

                record.update({
                    "numerical_assessment":
                        numeric["numerical_assessment"],
                    "numerical_reason":
                        numeric.get("reason", ""),
                    "numerical_needs_human_review":
                        numeric.get("needs_human_review", True),

                    "semantic_assessment":
                        semantic["semantic_assessment"],
                    "semantic_reason":
                        semantic.get("reason", ""),
                    "semantic_needs_human_review":
                        semantic.get("needs_human_review", True),

                    "rejected": is_rejected,
                    "final_category": final_category,
                    "status": "completed"
                })

            except Exception as e:
                record["error"] = str(e)
                record["status"] = "error"

            out.write(
                json.dumps(record, ensure_ascii=False, default=str) + "\n"
            )
            out.flush()

            processed += 1

            if record["status"] == "completed":
                completed_keys.add(key)
                print(
                    f"[{processed}/{total}] "
                    f"{question_id} | {condition} | "
                    f"{record['final_category']}"
                )
            else:
                print(
                    f"[{processed}/{total}] ERROR "
                    f"{question_id} | {condition}: "
                    f"{record.get('error', 'Unknown error')}"
                )

print("\nEvaluation pass finished.")
print("Results saved to:", OUTPUT_FILE)
print("Completed records:", len(completed_keys), "/", total)

Total responses: 300
Already completed: 0
Remaining: 300
Output file: evaluation_results.jsonl
[1/300] financebench_id_03029 | baseline | Partially Correct
[2/300] financebench_id_03029 | evidence_grounded | Correct
[3/300] financebench_id_04672 | baseline | Partially Correct
[4/300] financebench_id_04672 | evidence_grounded | Correct
[5/300] financebench_id_00499 | baseline | Incorrect
[6/300] financebench_id_00499 | evidence_grounded | Correct
[7/300] financebench_id_01226 | baseline | Incorrect
[8/300] financebench_id_01226 | evidence_grounded | Correct
[9/300] financebench_id_01865 | baseline | Incorrect
[10/300] financebench_id_01865 | evidence_grounded | Correct
[11/300] financebench_id_00807 | baseline | Incorrect
[12/300] financebench_id_00807 | evidence_grounded | Incorrect
[13/300] financebench_id_00941 | baseline | Incorrect
[14/300] financebench_id_00941 | evidence_grounded | Correct
[15/300] financebench_id_01858 | baseline | Correct
[16/300] financebench_id_01858 | eviden

In [1]:

import json
import time
import requests
from pathlib import Path

# Retry ONLY these five failed baseline records
FAILED_IDS = {
    "financebench_id_03882",
    "financebench_id_01935",
    "financebench_id_00684",
    "financebench_id_00288",
    "financebench_id_00299",
}

SOURCE_FILE = Path("evaluation_results.jsonl")
RETRY_FILE = Path("failed_baseline_retries.jsonl")

# Read the existing 300 results; do not rerun successful evaluations
with SOURCE_FILE.open("r", encoding="utf-8") as f:
    all_results = [json.loads(line) for line in f if line.strip()]

failed_records = [
    row for row in all_results
    if row.get("financebench_id") in FAILED_IDS
    and row.get("condition") == "baseline"
    and row.get("status") == "error"
]

print("Failed records found:", len(failed_records))
print("IDs:", [r["financebench_id"] for r in failed_records])

assert len(failed_records) == 5, (
    "Expected exactly 5 failed baseline records. "
    "Check SOURCE_FILE and the existing results before continuing."
)

def call_ollama_retry(prompt, attempts=3):
    """Retry the evaluator request without rerunning the original answer."""
    last_error = None

    for attempt in range(1, attempts + 1):
        try:
            response = requests.post(
                "http://localhost:11434/api/generate",
                json={
                    "model": "qwen3:4b",
                    "prompt": prompt,
                    "stream": False,
                    "format": "json",
                    "options": {"temperature": 0}
                },
                timeout=300
            )
            response.raise_for_status()
            data = response.json()
            text = data.get("response", "").strip()

            # Some Ollama/model responses may not parse as JSON.
            if not text:
                raise ValueError("Ollama returned an empty response.")

            parsed = json.loads(text)
            if not isinstance(parsed, dict):
                raise ValueError("Expected a JSON object.")

            return parsed

        except Exception as exc:
            last_error = exc
            print(f"  Attempt {attempt}/{attempts} failed: {exc}")
            if attempt < attempts:
                time.sleep(2)

    raise RuntimeError(f"All retry attempts failed: {last_error}")


def assess_numeric(row):
    prompt = f"""
You are evaluating the NUMERICAL correctness of a financial QA response.

Return ONLY a JSON object with exactly these keys:
- numerical_assessment: "Correct", "Incorrect", or "N/A"
- reason: concise explanation
- needs_human_review: true or false

Use "N/A" if the question/response makes no numerical claim that needs
checking. Compare units, period, entity, metric, and rounding carefully.
Do not confuse a different financial metric with the requested one.
If evidence is insufficient or the assessment is ambiguous, set
needs_human_review to true.

Question: {row["question"]}
Reference answer: {row["expected_answer"]}
Evidence: {row["evidence"]}
Response being evaluated: {row["response"]}
"""
    result = call_ollama_retry(prompt)

    if result.get("numerical_assessment") not in {
        "Correct", "Incorrect", "N/A"
    }:
        raise ValueError("Invalid numerical_assessment label.")

    return {
        "assessment": result["numerical_assessment"],
        "reason": result.get("reason", ""),
        "needs_human_review": bool(result.get("needs_human_review", False))
    }


def assess_semantic(row):
    prompt = f"""
You are evaluating the SEMANTIC correctness of a financial QA response.

Return ONLY a JSON object with exactly these keys:
- semantic_assessment: "Correct" or "Incorrect"
- reason: concise explanation
- needs_human_review: true or false

Judge whether the response answers the actual question for the correct
company/entity, metric/event, and reporting period. Do not require exact
wording. A numerically correct value for the wrong metric or entity is
not semantically correct. If the evidence is insufficient or the decision
is ambiguous, set needs_human_review to true.

Question: {row["question"]}
Reference answer: {row["expected_answer"]}
Evidence: {row["evidence"]}
Response being evaluated: {row["response"]}
"""
    result = call_ollama_retry(prompt)

    if result.get("semantic_assessment") not in {"Correct", "Incorrect"}:
        raise ValueError("Invalid semantic_assessment label.")

    return {
        "assessment": result["semantic_assessment"],
        "reason": result.get("reason", ""),
        "needs_human_review": bool(result.get("needs_human_review", False))
    }


def final_category(numerical, semantic, rejected=False):
    if rejected:
        return "Rejected"

    # N/A means there is no applicable numerical claim to score.
    if numerical == "N/A":
        return "Correct" if semantic == "Correct" else "Incorrect"

    if numerical == "Correct" and semantic == "Correct":
        return "Correct"
    if numerical == "Incorrect" and semantic == "Incorrect":
        return "Incorrect"
    return "Partially Correct"


# Write retries separately. The original 300-record file remains untouched.
retry_results = []

for index, row in enumerate(failed_records, start=1):
    print(f"\n[{index}/5] Retrying {row['financebench_id']}")

    try:
        numeric = assess_numeric(row)
        semantic = assess_semantic(row)

        result = {
            "financebench_id": row["financebench_id"],
            "condition": "baseline",
            "question": row["question"],
            "expected_answer": row["expected_answer"],
            "evidence": row["evidence"],
            "response": row["response"],
            "status": "completed",
            "numerical_assessment": numeric["assessment"],
            "numerical_reason": numeric["reason"],
            "numerical_needs_human_review": numeric["needs_human_review"],
            "semantic_assessment": semantic["assessment"],
            "semantic_reason": semantic["reason"],
            "semantic_needs_human_review": semantic["needs_human_review"],
            "rejected": False,
            "final_category": final_category(
                numeric["assessment"], semantic["assessment"]
            )
        }

    except Exception as exc:
        # Preserve failures for a later targeted retry; don't lose the record.
        result = {
            "financebench_id": row["financebench_id"],
            "condition": "baseline",
            "question": row["question"],
            "expected_answer": row["expected_answer"],
            "evidence": row["evidence"],
            "response": row["response"],
            "status": "error",
            "error": str(exc)
        }

    retry_results.append(result)
    print("Status:", result["status"])
    if result["status"] == "completed":
        print("Category:", result["final_category"])

with RETRY_FILE.open("w", encoding="utf-8") as f:
    for result in retry_results:
        f.write(json.dumps(result, ensure_ascii=False) + "\n")

print("\nRetry file saved to:", RETRY_FILE.resolve())
print("Completed:", sum(r["status"] == "completed" for r in retry_results))
print("Still failed:", sum(r["status"] == "error" for r in retry_results))


Failed records found: 5
IDs: ['financebench_id_03882', 'financebench_id_01935', 'financebench_id_00684', 'financebench_id_00288', 'financebench_id_00299']

[1/5] Retrying financebench_id_03882
  Attempt 1/3 failed: Ollama returned an empty response.
  Attempt 2/3 failed: Ollama returned an empty response.
  Attempt 3/3 failed: Ollama returned an empty response.
Status: error

[2/5] Retrying financebench_id_01935
  Attempt 1/3 failed: Ollama returned an empty response.
  Attempt 2/3 failed: Ollama returned an empty response.
  Attempt 3/3 failed: Ollama returned an empty response.
Status: error

[3/5] Retrying financebench_id_00684
  Attempt 1/3 failed: Ollama returned an empty response.
  Attempt 2/3 failed: Ollama returned an empty response.
  Attempt 3/3 failed: Ollama returned an empty response.
Status: error

[4/5] Retrying financebench_id_00288
  Attempt 1/3 failed: Ollama returned an empty response.
  Attempt 2/3 failed: Ollama returned an empty response.
  Attempt 3/3 failed: Ol

In [3]:

import json
import re
import requests

def call_ollama_retry(prompt, attempts=3):
    last_error = None

    for attempt in range(1, attempts + 1):
        try:
            r = requests.post(
                "http://localhost:11434/api/generate",
                json={
                    "model": "qwen3:4b",
                    "prompt": prompt,
                    "stream": False,
                    "think": False,
                    "format": "json",
                    "options": {"temperature": 0}
                },
                timeout=300
            )
            r.raise_for_status()
            data = r.json()

            # Prefer the normal answer; fall back to thinking if necessary.
            text = (data.get("response") or "").strip()
            if not text:
                text = (data.get("thinking") or "").strip()

            if not text:
                raise ValueError(
                    f"Both response and thinking were empty. "
                    f"done_reason={data.get('done_reason')}, "
                    f"total_duration={data.get('total_duration')}"
                )

            # Accept JSON surrounded by extra text, if any.
            match = re.search(r"\{.*\}", text, flags=re.DOTALL)
            if not match:
                raise ValueError(f"No JSON object found. Output: {text[:300]}")

            result = json.loads(match.group(0))
            if not isinstance(result, dict):
                raise ValueError("Expected a JSON object.")

            return result

        except Exception as exc:
            last_error = exc
            print(f"Attempt {attempt}/{attempts} failed: {exc}")

    raise RuntimeError(f"All attempts failed: {last_error}")


# Test just ONE of the failed records. Nothing is written to disk.
test_row = failed_records[0]
print("Testing:", test_row["financebench_id"])

try:
    numeric_test = assess_numeric(test_row)
    print("\nNumerical evaluation:")
    print(json.dumps(numeric_test, indent=2))

    semantic_test = assess_semantic(test_row)
    print("\nSemantic evaluation:")
    print(json.dumps(semantic_test, indent=2))

    print("\nTest passed — both evaluations returned valid results.")

except Exception as e:
    print("\nTest failed:", type(e).__name__, str(e))


Testing: financebench_id_03882

Numerical evaluation:
{
  "assessment": "Incorrect",
  "reason": "The reference answer $1616.00 is the 2020 trade receivables, net value from the balance sheet, which is the correct metric for net AR (accounts receivable) as per the question's request for 'net AR'. The response incorrectly states it's not determinable without specific figures, but the balance sheet explicitly shows trade receivables, net for 2020 as $1,615.9 million (rounded to $1,616.00 million).",
  "needs_human_review": false
}

Semantic evaluation:
{
  "assessment": "Incorrect",
  "reason": "The response claims it's not determinable from the balance sheet alone, but the balance sheet explicitly shows 'Trade receivables, net' as $1,615.9 million for 2020, which is the net AR figure directly stated in the balance sheet. The response incorrectly states that net AR requires gross AR minus allowance, but the balance sheet already provides net receivables without needing additional calcula

In [4]:

# Re-evaluate only the five previously failed baseline records.
# Uses the fixed call_ollama_retry() helper from the previous cell.

retry_results = []

for index, row in enumerate(failed_records, start=1):
    print(f"\n[{index}/5] Evaluating {row['financebench_id']}")

    try:
        numeric = assess_numeric(row)
        semantic = assess_semantic(row)

        result = {
            "financebench_id": row["financebench_id"],
            "condition": "baseline",
            "question": row["question"],
            "expected_answer": row["expected_answer"],
            "evidence": row["evidence"],
            "response": row["response"],
            "status": "completed",
            "numerical_assessment": numeric["assessment"],
            "numerical_reason": numeric["reason"],
            "numerical_needs_human_review": numeric["needs_human_review"],
            "semantic_assessment": semantic["assessment"],
            "semantic_reason": semantic["reason"],
            "semantic_needs_human_review": semantic["needs_human_review"],
            "rejected": False,
            "final_category": final_category(
                numeric["assessment"],
                semantic["assessment"]
            )
        }

    except Exception as exc:
        result = {
            "financebench_id": row["financebench_id"],
            "condition": "baseline",
            "status": "error",
            "error": str(exc)
        }

    retry_results.append(result)
    print("Status:", result["status"])

    if result["status"] == "completed":
        print("Numerical:", result["numerical_assessment"])
        print("Semantic:", result["semantic_assessment"])
        print("Final category:", result["final_category"])

# Save the retry results separately; preserve all original results.
with open("failed_baseline_retries.jsonl", "w", encoding="utf-8") as f:
    for result in retry_results:
        f.write(json.dumps(result, ensure_ascii=False) + "\n")

print("\nCompleted:", sum(r["status"] == "completed" for r in retry_results))
print("Still failed:", sum(r["status"] == "error" for r in retry_results))
print("Saved to failed_baseline_retries.jsonl")



[1/5] Evaluating financebench_id_03882
Status: completed
Numerical: Incorrect
Semantic: Incorrect
Final category: Incorrect

[2/5] Evaluating financebench_id_01935
Status: completed
Numerical: Incorrect
Semantic: Incorrect
Final category: Incorrect

[3/5] Evaluating financebench_id_00684
Status: completed
Numerical: Correct
Semantic: Correct
Final category: Correct

[4/5] Evaluating financebench_id_00288
Status: completed
Numerical: Incorrect
Semantic: Incorrect
Final category: Incorrect

[5/5] Evaluating financebench_id_00299
Status: completed
Numerical: Incorrect
Semantic: Incorrect
Final category: Incorrect

Completed: 5
Still failed: 0
Saved to failed_baseline_retries.jsonl


In [5]:

import json
from pathlib import Path
from collections import Counter

SOURCE_FILE = Path("evaluation_results.jsonl")
RETRY_FILE = Path("failed_baseline_retries.jsonl")
FINAL_FILE = Path("evaluation_results_final.jsonl")

# Load the original results
with SOURCE_FILE.open("r", encoding="utf-8") as f:
    original = [json.loads(line) for line in f if line.strip()]

# Load the five retry results
with RETRY_FILE.open("r", encoding="utf-8") as f:
    retries = [json.loads(line) for line in f if line.strip()]

# Only merge successful retries
successful_retries = {
    (r["financebench_id"], r["condition"]): r
    for r in retries
    if r.get("status") == "completed"
}

assert len(original) == 300, f"Expected 300 original rows, found {len(original)}"
assert len(successful_retries) == 5, (
    f"Expected 5 successful retries, found {len(successful_retries)}"
)

# Replace matching original records; keep all other rows unchanged.
merged = []
replaced = set()

for row in original:
    key = (row["financebench_id"], row["condition"])

    if key in successful_retries:
        assert row.get("status") == "error", (
            f"Refusing to replace a non-error record: {key}"
        )
        merged.append(successful_retries[key])
        replaced.add(key)
    else:
        merged.append(row)

assert replaced == set(successful_retries), (
    "Some retry records did not match original error records."
)

# Validate row counts and unique evaluation keys
keys = [(r["financebench_id"], r["condition"]) for r in merged]
assert len(merged) == 300
assert len(set(keys)) == 300, "Duplicate question-condition pairs found!"

# Save the merged file
with FINAL_FILE.open("w", encoding="utf-8") as f:
    for row in merged:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

# Report final status and category counts
status_counts = Counter(r.get("status", "missing") for r in merged)
condition_counts = Counter(r["condition"] for r in merged)
category_counts = {
    condition: Counter(
        r.get("final_category")
        for r in merged
        if r["condition"] == condition and r.get("status") == "completed"
    )
    for condition in condition_counts
}

print("FINAL FILE:", FINAL_FILE.resolve())
print("Total records:", len(merged))
print("Unique question-condition pairs:", len(set(keys)))
print("Status counts:", dict(status_counts))
print("\nFinal categories:")
for condition, counts in category_counts.items():
    print(condition, dict(counts))


FINAL FILE: /home/arav/VS-Code/financebench-safety/evaluation_results_final.jsonl
Total records: 300
Unique question-condition pairs: 300
Status counts: {'completed': 300}

Final categories:
baseline {'Partially Correct': 41, 'Incorrect': 92, 'Correct': 17}
evidence_grounded {'Correct': 118, 'Incorrect': 11, 'Partially Correct': 21}
